# 🔍 高清修复：固定主体，重绘背景

以 `photo/_verify_output.png`（01 生成的宇航员猫）为例：**宇航员保持不变**，把周围的太空背景重绘成 768×768 高清版。

**实现思路（img2img + 掩膜贴回）**：
1. 原图放大到 768×768（主体变清晰底版）
2. 做掩膜 mask：白色=保护区域（宇航员），黑色=重绘区域（太空），边缘羽化让过渡自然
3. img2img 整图重绘（背景变高清太空）
4. 用 mask 把原图主体贴回 —— **主体像素 100% 不变**，背景是重绘的高清版

**为什么不用官方 Inpaint 管线？** `StableDiffusionInpaintPipeline` 要求 UNet 是 9 通道（inpaint 专用版），本地 `stable-diffusion-v1-5` 的 UNet 是 4 通道（文生图版），所以用上面这套组合达到同样效果。

请选择内核：**transformers (15-transformers-basics)**

In [ ]:
# 1. 加载 img2img 管线（与 02 相同的本地模型）
import torch
from diffusers import StableDiffusionImg2ImgPipeline

MODEL_DIR = r'S:\PythonAILearningCourse\04-ai-painting-sdxl\stable-diffusion-v1-5'

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,
    variant='fp16',
)
pipe.to('cuda')
pipe.enable_attention_slicing()
print('✅ 管线加载完成')

In [ ]:
# 2. 载入原图 + 制作掩膜（白=保护宇航员，黑=重绘太空）
from PIL import Image, ImageDraw, ImageFilter

SRC = r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\_verify_output.png'
SIZE = 768   # 修复后的分辨率；SD1.5 用 768 比 1024 稳（1024 易出现内容重复）

# 原图放大（Lanczos 插值，主体清晰的底版）
init_image = Image.open(SRC).convert('RGB').resize((SIZE, SIZE), Image.LANCZOS)

# ---- 掩膜：可根据主体位置调整椭圆参数 ----
mask = Image.new('L', (SIZE, SIZE), 0)            # 全黑 = 默认全部重绘
draw = ImageDraw.Draw(mask)
cx, cy = SIZE // 2, int(SIZE * 0.47)              # 椭圆中心（宇航员大概在中间）
rx, ry = int(SIZE * 0.26), int(SIZE * 0.38)       # 椭圆半径（罩住整个宇航员）
draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=255)   # 白色 = 保护区
FEATHER = 15                                      # 羽化半径：边缘过渡更自然
mask = mask.filter(ImageFilter.GaussianBlur(FEATHER))

print('掩膜（白色区域会被固定）：')
mask

In [ ]:
# 3. img2img 重绘：prompt 只描述背景太空（宇航员会被贴回，不受 prompt 影响）
repainted = pipe(
    prompt='deep space starfield, colorful nebula, cosmic dust, bright stars, sci-fi, highly detailed, sharp focus',
    negative_prompt='lowres, blurry, watermark, text, cat, astronaut',
    image=init_image,
    strength=0.55,             # 背景重绘幅度：越大变化越大（0.4~0.7 之间调）
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator('cuda').manual_seed(42),
).images[0]

print('img2img 重绘结果（注意：此时宇航员也被改动了）：')
repainted

In [ ]:
# 4. 贴回合成：mask 白色区域取原图（宇航员原封不动），黑色区域取重绘背景
from PIL import Image

final = Image.composite(init_image, repainted, mask)
final.save(r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\hifix_output.png')

# 原图 vs 修复结果 左右对比（512 缩略显示）
combo = Image.new('RGB', (SIZE, SIZE // 2))
combo.paste(Image.open(SRC).convert('RGB').resize((SIZE // 2, SIZE // 2)), (0, 0))
combo.paste(final.resize((SIZE // 2, SIZE // 2)), (SIZE // 2, 0))
combo.save(r'S:\PythonAILearningCourse\04-ai-painting-sdxl\photo\hifix_compare.png')

print('✅ 修复完成（左：原图 512，右：修复后 768）：')
combo